# RAG Assignment: Baseline vs. Cross-Encoder Re-Ranking

Two RAG pipelines over one chunked PDF (IRS Publication 501, 2025). They share the same bi-encoder Top-10 and the same LLM; only the three chunks given to the LLM differ:

* **Pipeline 1 (baseline):** bi-encoder Top-10 → Top-3 as ranked → LLM → Answer 1
* **Pipeline 2 (re-ranking):** same Top-10 → Cross-Encoder re-sorts → Top-3 → same LLM → Answer 2

This notebook holds the **code and the raw results**. The explanation, the failure-case analysis and the conclusions are in the report, **`report/report.pdf`**, whose section numbers are referenced below.

Every part of the implementation is defined inline in the code cells (marked `# === Implementation: … ===`). The same code is kept in `src/` for the unit tests, and a test checks that the two are identical.

## 1. Setup and configuration

In [1]:
# Standard library
import hashlib, json, os, platform, re, statistics, time, warnings
from dataclasses import asdict, dataclass, field
from typing import Callable, Dict, List, Optional, Sequence
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Run from the repository root regardless of where Jupyter was started.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

try:
    from dotenv import load_dotenv
    load_dotenv(".env")    # picks up LLM_BACKEND / API keys from .env if present
except ImportError:
    pass

# Third-party: numerics, models (sentence-transformers), vector search (FAISS), PDF parsing (PyMuPDF)
import numpy as np, pandas as pd, torch
import sentence_transformers, transformers, faiss, pymupdf

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
torch.manual_seed(42); np.random.seed(42)   # reproducibility
os.makedirs("results", exist_ok=True)

env_info = {
    "python": platform.python_version(), "platform": platform.platform(),
    "cpu": platform.processor() or platform.machine(), "torch_threads": torch.get_num_threads(),
    "cuda": torch.cuda.is_available(), "torch": torch.__version__,
    "sentence_transformers": sentence_transformers.__version__, "transformers": transformers.__version__,
    "faiss": faiss.__version__, "pymupdf": pymupdf.__version__,
}
print(json.dumps(env_info, indent=1))

{
 "python": "3.10.12",
 "platform": "Linux-6.17.9-76061709-generic-x86_64-with-glibc2.35",
 "cpu": "x86_64",
 "torch_threads": 12,
 "cuda": false,
 "torch": "2.7.0+cpu",
 "sentence_transformers": "5.1.2",
 "transformers": "4.57.1",
 "faiss": "1.12.0",
 "pymupdf": "1.26.5"
}


In [2]:
# === Implementation: Configuration ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

@dataclass
class Config:
    pdf_path: str = "data/p501.pdf"
    chunk_size: int = 150          # words per chunk
    chunk_overlap: int = 30        # words shared by consecutive chunks
    embed_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    cross_encoder_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    initial_k: int = 10            # candidates retrieved by the bi-encoder
    final_k: int = 3               # chunks passed to the LLM
    llm_backend: str = "hf"        # hf | ollama | openai | gemini | anthropic
    llm_model: str = "Qwen/Qwen2.5-1.5B-Instruct"
    max_new_tokens: int = 256
    temperature: float = 0.0       # 0 => greedy / deterministic decoding
    seed: int = 42

    def validate(self) -> "Config":
        if self.chunk_size <= 0:
            raise ValueError("chunk_size must be positive")
        if not 0 <= self.chunk_overlap < self.chunk_size:
            raise ValueError("chunk_overlap must be in [0, chunk_size)")
        if self.final_k <= 0 or self.initial_k <= 0:
            raise ValueError("initial_k and final_k must be positive")
        if self.final_k > self.initial_k:
            raise ValueError("final_k cannot exceed initial_k")
        if self.llm_backend not in {"hf", "ollama", "openai", "gemini", "anthropic"}:
            raise ValueError(f"unknown llm_backend {self.llm_backend!r}")
        return self

    @classmethod
    def from_env(cls, **overrides) -> "Config":
        """Read optional overrides from environment variables (see .env.example)."""
        env = {
            "llm_backend": os.getenv("LLM_BACKEND"),
            "llm_model": os.getenv("LLM_MODEL"),
        }
        kwargs = {k: v for k, v in env.items() if v}
        kwargs.update(overrides)
        return cls(**kwargs).validate()

In [3]:
# === Implementation: Utilities ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

def save_json(obj, path: str) -> None:
    """Write experiment outputs to results/ (dataclasses such as Chunk are serialised as dicts)."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=lambda o: asdict(o) if hasattr(o, "__dataclass_fields__") else str(o))

In [4]:
# Default settings, optionally overridden by LLM_BACKEND / LLM_MODEL in .env
cfg = Config.from_env()
print(json.dumps(cfg.__dict__, indent=1))

{
 "pdf_path": "data/p501.pdf",
 "chunk_size": 150,
 "chunk_overlap": 30,
 "embed_model": "sentence-transformers/all-MiniLM-L6-v2",
 "cross_encoder_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
 "initial_k": 10,
 "final_k": 3,
 "llm_backend": "gemini",
 "llm_model": "gemini-3.5-flash",
 "max_new_tokens": 256,
 "temperature": 0.0,
 "seed": 42
}


## 2. Source document

IRS Publication 501 (2025), 31 pages, https://www.irs.gov/pub/irs-pdf/p501.pdf (U.S. government work, public domain). The cell downloads it if it is missing and records its SHA-256. Why this document was chosen: report §1.

In [5]:
PDF_URL = "https://www.irs.gov/pub/irs-pdf/p501.pdf"
if not os.path.exists(cfg.pdf_path):
    import requests
    os.makedirs(os.path.dirname(cfg.pdf_path), exist_ok=True)
    open(cfg.pdf_path, "wb").write(requests.get(PDF_URL, timeout=60).content)

with pymupdf.open(cfg.pdf_path) as d:
    meta, n_pages = d.metadata, d.page_count
source = {"title": meta.get("title"), "subject": meta.get("subject").strip(), "url": PDF_URL,
          "pages": n_pages, "creationDate": meta.get("creationDate"),
          "sha256": hashlib.sha256(open(cfg.pdf_path, "rb").read()).hexdigest()}  # fingerprint of the exact file used
source

{'title': '2025 Publication 501',
 'subject': 'Dependents, Standard Deduction, and Filing Information',
 'url': 'https://www.irs.gov/pub/irs-pdf/p501.pdf',
 'pages': 31,
 'creationDate': "D:20260204142345-05'00'",
 'sha256': '288b62a4917f3bc3df0bade7a22c23a8ae648ba9eb5086df5b8b50e881148ff6'}

## 3. Extraction and cleaning

Read each page with PyMuPDF, remove the footer, page numbers and margin icons, and keep the page number for every page.

In [6]:
# === Implementation: PDF extraction and cleaning ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

# Running footer ("Publication 501 (2025)" followed by the page number),
# margin icons and bullet glyphs that pymupdf emits on their own lines.
_NOISE_LINE = re.compile(r"^(Publication 501 \(\d{4}\)|Page \d+|\d{1,2}|CAUTION|TIP|!|•|▲|▼)$")


def clean_page_text(text: str) -> str:
    """Turn one page of raw PDF text into a single clean line of prose."""
    lines = [ln.strip() for ln in text.splitlines()]
    # drop empty lines and layout noise (footer, page number, CAUTION/TIP icons, bullets)
    kept = [ln for ln in lines if ln and not _NOISE_LINE.match(ln)]
    out = " ".join(kept)
    # normalise curly quotes so string matching against the PDF is reliable
    out = out.replace("’", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", out).strip()


def extract_pages(pdf_path: str) -> List[Dict]:
    """Return [{'page': 1-based page number, 'text': cleaned text}, ...]."""
    import pymupdf

    pages = []
    with pymupdf.open(pdf_path) as doc:
        # TEXT_DEHYPHENATE re-joins words split across lines ("fil-\ning" -> "filing");
        # PyMuPDF already returns the two-column layout in reading order.
        flags = pymupdf.TEXTFLAGS_TEXT | pymupdf.TEXT_DEHYPHENATE
        for i, page in enumerate(doc):
            pages.append({"page": i + 1, "text": clean_page_text(page.get_text(flags=flags))})
    return pages


def file_sha256(path: str) -> str:
    """Fingerprint of the source PDF, recorded so the exact file can be verified."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 16), b""):
            h.update(block)
    return h.hexdigest()

In [7]:
pages = extract_pages(cfg.pdf_path)
print(f"{len(pages)} pages, {sum(len(p['text'].split()) for p in pages):,} words after cleaning")
print("\nPage 3 excerpt:\n", pages[2]["text"][600:1400])

31 pages, 34,973 words after cleaning

Page 3 excerpt:
 iling requirements apply even if you owe no tax. You may have to pay a penalty if you are required to file a return but fail to do so. If you willfully fail to file a return, you may be subject to criminal prosecution. Gross income. Gross income is all income you receive in the form of money, goods, property, and services that isn't exempt from tax. If you are married and live with your spouse in a community property state, half of any income defined by state law as community income may be considered yours. For a list of community property states, see Community property states under Married Filing Separately, later. Self-employed persons. If you are self-employed in a business that provides services (where products aren't a factor), your gross income from that business is the gross receipt


## 4. Chunking

150-word sliding window with 30 words of overlap, across page breaks. Each chunk records its page range and gets a stable ID (`C000`…). The output also checks that no chunk exceeds the bi-encoder's 256-token limit.

In [8]:
# === Implementation: Chunking ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

@dataclass
class Chunk:
    """One retrievable unit of text plus the PDF pages it came from."""
    chunk_id: str
    text: str
    page_start: int
    page_end: int
    word_start: int  # offset of first word in the whole-document word stream

    @property
    def pages(self) -> str:
        return f"p.{self.page_start}" if self.page_start == self.page_end else f"pp.{self.page_start}-{self.page_end}"


def chunk_pages(pages: Sequence[Dict], chunk_size: int, chunk_overlap: int) -> List[Chunk]:
    """Sliding word window over the whole document.

    Every word remembers the page it came from, so a chunk that crosses a
    page break records both the first and the last page.  IDs are assigned
    in document order and are therefore stable for a fixed config + PDF.
    """
    if chunk_size <= 0 or not 0 <= chunk_overlap < chunk_size:
        raise ValueError("invalid chunk_size / chunk_overlap")
    # 1) flatten the document into one word stream, remembering each word's page
    words, word_pages = [], []
    for p in pages:
        for w in p["text"].split():
            words.append(w)
            word_pages.append(p["page"])
    # 2) slide a window of `chunk_size` words, advancing by size - overlap,
    #    so consecutive chunks share `chunk_overlap` words
    chunks: List[Chunk] = []
    step = chunk_size - chunk_overlap
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        # page range = page of the first word .. page of the last word
        chunks.append(Chunk(
            chunk_id=f"C{len(chunks):03d}",
            text=" ".join(words[start:end]),
            page_start=word_pages[start],
            page_end=word_pages[end - 1],
            word_start=start,
        ))
        if end == len(words):
            break
        start += step
    return chunks

In [9]:
chunks = chunk_pages(pages, cfg.chunk_size, cfg.chunk_overlap)
lens = [len(c.text.split()) for c in chunks]
print(f"{len(chunks)} chunks | words per chunk: min {min(lens)}, median {int(np.median(lens))}, max {max(lens)}")
print(f"chunks spanning a page break: {sum(c.page_start != c.page_end for c in chunks)}")
tok = transformers.AutoTokenizer.from_pretrained(cfg.embed_model)
ntok = [len(tok(c.text)["input_ids"]) for c in chunks]
print(f"bi-encoder tokens per chunk: median {int(np.median(ntok))}, max {max(ntok)} "
      f"(model limit 256; truncated chunks: {sum(n > 256 for n in ntok)})")
save_json([c.__dict__ for c in chunks], "results/chunks.json")
pd.DataFrame([{"chunk_id": c.chunk_id, "pages": c.pages, "text": c.text[:110] + "…"} for c in chunks[13:17]])

292 chunks | words per chunk: min 53, median 150, max 150
chunks spanning a page break: 36


bi-encoder tokens per chunk: median 192, max 255 (model limit 256; truncated chunks: 0)


,chunk_id,pages,text
0,C013,pp.2-3,"but not losses, reported on Form 8949 or Schedule D. Gross income from a business means, for example, the amou…"
1,C014,p.3,by Custodial Parent 8814 Parents' Election To Report Child's Interest and Dividends Who Must File If you are a…
2,C015,p.3,"is all income you receive in the form of money, goods, property, and services that isn't exempt from tax. If y…"
3,C016,p.3,"either case, you must add any income from investments and from incidental or outside operations or sources. Fi…"


## 5. Stage 1 — bi-encoder index

`all-MiniLM-L6-v2` embeds every chunk once; normalised vectors in a FAISS inner-product index give cosine similarity. (Architecture comparison: report §2.)

In [10]:
# === Implementation: Bi-encoder index (Stage 1) ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

class BiEncoderIndex:
    """Dense index: normalised embeddings + FAISS inner product (= cosine)."""

    def __init__(self, model_name: str, chunks: Sequence[Chunk], device: str = "cpu"):
        import faiss
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(model_name, device=device)
        self.chunks = list(chunks)
        # Offline step: embed every chunk ONCE.  normalize_embeddings=True makes
        # every vector unit length, so inner product == cosine similarity.
        emb = self.model.encode([c.text for c in self.chunks], batch_size=32,
                                normalize_embeddings=True, convert_to_numpy=True,
                                show_progress_bar=False).astype("float32")
        self.embeddings = emb
        # exact (brute-force) inner-product index; fine for a few hundred vectors
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb)

    def search(self, query: str, k: int) -> List[Dict]:
        """Stage 1 at query time: embed the query independently of any chunk,
        then return the k chunks with the highest cosine similarity."""
        q = self.model.encode([query], normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
        scores, idx = self.index.search(q, k)
        # keep the original rank and score of every candidate - they are never overwritten
        return [
            {"initial_rank": r + 1, "chunk_id": self.chunks[i].chunk_id, "chunk_idx": int(i),
             "cosine": float(s), "pages": self.chunks[i].pages, "text": self.chunks[i].text}
            for r, (s, i) in enumerate(zip(scores[0], idx[0])) if i >= 0
        ]

In [11]:
t0 = time.perf_counter()
index = BiEncoderIndex(cfg.embed_model, chunks)
t_index = time.perf_counter() - t0
print(f"indexed {index.index.ntotal} chunks, dim={index.embeddings.shape[1]}, "
      f"model load + encode all chunks: {t_index:.1f}s")
print("norms ≈ 1:", np.allclose(np.linalg.norm(index.embeddings, axis=1), 1, atol=1e-4))

indexed 292 chunks, dim=384, model load + encode all chunks: 8.6s
norms ≈ 1: True


## 6. Stage 2 — Cross-Encoder

`cross-encoder/ms-marco-MiniLM-L-6-v2` scores each (query, chunk) pair jointly. The example below scores a relevant and an irrelevant passage.

In [12]:
# === Implementation: Cross-Encoder reranking (Stage 2) ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

class Reranker:
    """Stage 2: a Cross-Encoder reads (query, chunk) TOGETHER in one transformer
    pass and outputs one relevance logit.  Nothing can be pre-computed."""

    def __init__(self, model_name: str, device: str = "cpu"):
        from sentence_transformers import CrossEncoder

        self.model = CrossEncoder(model_name, device=device)

    def score(self, query: str, texts: Sequence[str]) -> List[float]:
        # one forward pass per (query, chunk) pair; the pairs are batched together
        return [float(s) for s in self.model.predict([(query, t) for t in texts],
                                                     show_progress_bar=False)]


def rerank_candidates(candidates: Sequence[Dict], ce_scores: Sequence[float]) -> List[Dict]:
    """Attach Cross-Encoder scores and return a *new* list sorted by them.

    The original ``initial_rank`` of every candidate is kept untouched; a new
    ``rerank_rank`` is added.  Ties are broken by the original rank so the
    ordering is deterministic.
    """
    if len(candidates) != len(ce_scores):
        raise ValueError("one Cross-Encoder score is required per candidate")
    # copy each candidate (so the baseline list is not modified) and attach its score
    scored = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    # sort by Cross-Encoder score, highest first
    scored.sort(key=lambda c: (-c["ce_score"], c["initial_rank"]))
    for r, c in enumerate(scored, start=1):
        c["rerank_rank"] = r
    return scored


def select_top(candidates: Sequence[Dict], k: int) -> List[Dict]:
    """The k chunks handed to the LLM: the first k of whichever ordering is given
    (bi-encoder order for the baseline, Cross-Encoder order for re-ranking)."""
    if k <= 0:
        raise ValueError("k must be positive")
    return list(candidates[:k])

In [13]:
reranker = Reranker(cfg.cross_encoder_model)
print(reranker.score("Can housekeepers be claimed as dependents?",
                     ["Housekeepers, maids, or servants. If these people work for you, you can't claim them as dependents.",
                      "Your home was the main home of your child for more than half the year."]))

[9.604876518249512, -11.156329154968262]


## 7. LLM and the shared prompt

Backend and model come from `.env` (reported run: `gemini-3.5-flash`, temperature 0, 256 tokens, thinking off). Both pipelines use the same system prompt and template, printed below. Answers are cached in `results/llm_cache.json`, so a re-run uses no API quota. Settings and the reasons for them: report §1 and §3.

In [14]:
# === Implementation: LLM backends — identical prompt for both pipelines ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

SYSTEM_PROMPT = (
    "You answer questions about IRS Publication 501 (2025) using ONLY the "
    "numbered context passages provided. If the passages do not contain the "
    "answer, say that the provided context is insufficient. Do not use outside "
    "knowledge or make unsupported claims. State any exceptions or conditions "
    "that the passages mention, and cite the page reference of each passage you "
    "use, e.g. (p. 7)."
)


def build_prompt(query: str, contexts: Sequence[Dict]) -> str:
    blocks = [f"[{i}] ({c['pages']}, {c['chunk_id']})\n{c['text']}" for i, c in enumerate(contexts, 1)]
    return "Context passages:\n\n" + "\n\n".join(blocks) + f"\n\nQuestion: {query}\nAnswer:"


class LLM:
    """Thin wrapper so every backend receives the same messages and settings."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.backend = cfg.llm_backend
        self.model_name = cfg.llm_model
        if self.backend == "hf":
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer

            torch.manual_seed(cfg.seed)
            # Prefer a copy under models/<name> (see README) over the HF cache.
            local = os.path.join("models", cfg.llm_model.split("/")[-1])
            src = local if os.path.isdir(local) else cfg.llm_model
            self.tok = AutoTokenizer.from_pretrained(src)
            self.model = AutoModelForCausalLM.from_pretrained(src, dtype=torch.bfloat16)
            self.model.eval()
        self._client = None  # API clients are created once, on first use, and reused
        # Optional answer cache (LLM_CACHE=path): a re-run reuses answers already
        # generated for the *identical* model + prompt, together with the timing
        # of the original call, so it does not burn free-tier API quota.
        self.cache_path = os.getenv("LLM_CACHE")
        self.cache = {}
        if self.cache_path and os.path.exists(self.cache_path):
            with open(self.cache_path) as f:
                self.cache = json.load(f)

    def cache_key(self, prompt: str) -> str:
        raw = json.dumps([self.backend, self.model_name, SYSTEM_PROMPT, prompt, self.cfg.temperature,
                          self.cfg.max_new_tokens])
        return hashlib.sha256(raw.encode()).hexdigest()

    def _api_client(self):
        if self._client is None:
            if self.backend == "gemini":
                from google import genai
                self._client = genai.Client()
            elif self.backend == "openai":
                from openai import OpenAI
                self._client = OpenAI()
            elif self.backend == "anthropic":
                import anthropic
                self._client = anthropic.Anthropic()
        return self._client

    def generate(self, prompt: str) -> str:
        """Call the backend.  For API backends, free-tier rate limits are handled
        by pacing calls and retrying 429/5xx errors with back-off.  The waiting
        is recorded separately (``last_wait_s``) so timings only count the
        successful call (``last_call_s``)."""
        self.last_wait_s = 0.0
        self.last_cached = False
        key = self.cache_key(prompt)
        if key in self.cache:
            hit = self.cache[key]
            self.last_call_s, self.last_cached = hit["call_s"], True
            return hit["answer"]
        out = self._generate_uncached(prompt)
        if self.cache_path:
            self.cache[key] = {"answer": out, "call_s": self.last_call_s, "model": self.model_name}
            with open(self.cache_path, "w") as f:
                json.dump(self.cache, f, indent=1)
        return out

    def _generate_uncached(self, prompt: str) -> str:
        self.last_wait_s = 0.0
        api = self.backend in {"gemini", "openai", "anthropic"}
        min_gap = float(os.getenv("API_MIN_INTERVAL_S", "4")) if api else 0.0
        for attempt in range(8):
            gap = min_gap - (time.perf_counter() - getattr(self, "_last_end", -1e9))
            if gap > 0:
                time.sleep(gap)
                self.last_wait_s += gap
            t0 = time.perf_counter()
            try:
                out = self._call(prompt)
                self.last_call_s = time.perf_counter() - t0
                self._last_end = time.perf_counter()
                return out
            except Exception as e:  # retry only rate-limit / transient server errors
                self._last_end = time.perf_counter()
                msg = str(e)
                if not api or attempt == 7 or not any(k in msg for k in ("429", "RESOURCE_EXHAUSTED", "500", "503", "UNAVAILABLE", "overloaded")):
                    raise
                back = min(60.0, 5.0 * 2 ** attempt)
                time.sleep(back)
                self.last_wait_s += (time.perf_counter() - t0)  # failed call + back-off

    def _call(self, prompt: str) -> str:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
        b, cfg = self.backend, self.cfg
        if b == "hf":
            import torch

            ids = self.tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
            with torch.no_grad():
                out = self.model.generate(ids, attention_mask=torch.ones_like(ids),
                                          max_new_tokens=cfg.max_new_tokens, do_sample=False,
                                          pad_token_id=self.tok.eos_token_id)
            return self.tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
        if b == "ollama":
            import requests

            host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
            r = requests.post(f"{host}/api/chat", timeout=600, json={
                "model": self.model_name, "messages": msgs, "stream": False,
                "options": {"temperature": cfg.temperature, "seed": cfg.seed,
                            "num_predict": cfg.max_new_tokens}})
            r.raise_for_status()
            return r.json()["message"]["content"].strip()
        if b == "openai":
            r = self._api_client().chat.completions.create(model=self.model_name, messages=msgs,
                                                 temperature=cfg.temperature,
                                                 max_tokens=cfg.max_new_tokens, seed=cfg.seed)
            return r.choices[0].message.content.strip()
        if b == "anthropic":
            r = self._api_client().messages.create(model=self.model_name, system=SYSTEM_PROMPT,
                                                      messages=msgs[1:], temperature=cfg.temperature,
                                                      max_tokens=cfg.max_new_tokens)
            return r.content[0].text.strip()
        if b == "gemini":
            from google.genai import types

            r = self._api_client().models.generate_content(
                model=self.model_name, contents=prompt,
                config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT,
                                                   temperature=cfg.temperature, seed=cfg.seed,
                                                   max_output_tokens=cfg.max_new_tokens,
                                                   # no hidden "thinking": it would consume the
                                                   # output budget and make the step incomparable
                                                   thinking_config=types.ThinkingConfig(thinking_budget=0)))
            return (r.text or "").strip()
        raise ValueError(b)

In [15]:
llm = LLM(cfg)
print("SYSTEM PROMPT:\n" + SYSTEM_PROMPT)
print("\nUSER PROMPT TEMPLATE (example with one dummy passage):\n" +
      build_prompt("<question>", [{"pages": "p.N", "chunk_id": "C###", "text": "<chunk text>"}]))

SYSTEM PROMPT:
You answer questions about IRS Publication 501 (2025) using ONLY the numbered context passages provided. If the passages do not contain the answer, say that the provided context is insufficient. Do not use outside knowledge or make unsupported claims. State any exceptions or conditions that the passages mention, and cite the page reference of each passage you use, e.g. (p. 7).

USER PROMPT TEMPLATE (example with one dummy passage):
Context passages:

[1] (p.N, C###)
<chunk text>

Question: <question>
Answer:


## 8. Pipeline sanity checks

Assertions on a real query: one shared Top-10, original ranks kept, the baseline takes ranks 1–3, re-ranking takes the Cross-Encoder Top-3 from the same pool.

In [16]:
# === Implementation: The two pipelines ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

def _now() -> float:
    return time.perf_counter()  # monotonic, high resolution


@dataclass
class RAGSystem:
    """Holds the shared components and runs Pipeline 1 and Pipeline 2 side by side."""
    cfg: Config
    index: BiEncoderIndex
    reranker: Reranker
    llm: Optional[LLM] = None

    def retrieve(self, query: str):
        """Stage 1 (both pipelines): bi-encoder Top-10, timed."""
        t0 = _now()
        cands = self.index.search(query, self.cfg.initial_k)
        return cands, _now() - t0

    def rerank(self, query: str, cands: Sequence[Dict]):
        """Stage 2 (Pipeline 2 only): Cross-Encoder over the same candidates, timed."""
        t0 = _now()
        scores = self.reranker.score(query, [c["text"] for c in cands])
        reranked = rerank_candidates(cands, scores)
        return reranked, _now() - t0

    def compare(self, query: str, generate: bool = True) -> Dict:
        """Run both pipelines on ONE shared Top-10 candidate pool."""
        # Stage 1 runs once; both pipelines start from exactly this list
        cands, t_ret = self.retrieve(query)
        # Stage 2 re-orders the same 10 candidates (it cannot add new ones)
        reranked, t_rr = self.rerank(query, cands)
        base_top = select_top(cands, self.cfg.final_k)     # Pipeline 1: bi-encoder Top-3
        rr_top = select_top(reranked, self.cfg.final_k)    # Pipeline 2: Cross-Encoder Top-3
        # one table in ORIGINAL order with both ranks/scores side by side
        ce_by_id = {c["chunk_id"]: c for c in reranked}
        table = [dict(c, ce_score=ce_by_id[c["chunk_id"]]["ce_score"],
                      rerank_rank=ce_by_id[c["chunk_id"]]["rerank_rank"]) for c in cands]
        res = {
            "query": query,
            "candidates": [{k: v for k, v in c.items() if k != "text"} for c in table],
            "baseline_top3": [c["chunk_id"] for c in base_top],
            "rerank_top3": [c["chunk_id"] for c in rr_top],
            "timing_s": {"retrieval": t_ret, "rerank": t_rr},
        }
        if generate and self.llm is not None:
            for name, ctx in (("baseline", base_top), ("rerank", rr_top)):
                prompt = build_prompt(query, ctx)
                res[f"answer_{name}"] = self.llm.generate(prompt)
                res["timing_s"][f"generation_{name}"] = getattr(self.llm, "last_call_s", None)
                res[f"prompt_{name}"] = prompt
        return res


def promoted(result: Dict, final_k: int = 3) -> List[Dict]:
    """Candidates that were outside the baseline Top-k but entered the reranked Top-k."""
    return [c for c in result["candidates"] if c["initial_rank"] > final_k and c["rerank_rank"] <= final_k]

In [17]:
# Wire the shared components together: one index, one reranker, one LLM for both pipelines
system = RAGSystem(cfg, index, reranker, llm)

r = system.compare("Can a nonresident alien file a joint return?", generate=False)
cand = pd.DataFrame(r["candidates"])
assert len(cand) == cfg.initial_k == 10
assert list(cand.initial_rank) == list(range(1, 11))                       # original order preserved
assert r["baseline_top3"] == list(cand.sort_values("initial_rank").chunk_id[:3])
assert r["rerank_top3"] == list(cand.sort_values("rerank_rank").chunk_id[:3])
assert set(r["rerank_top3"]) <= set(cand.chunk_id)                         # reranker only reorders the pool
assert cand.cosine.is_monotonic_decreasing
print("all invariants hold"); cand[["initial_rank", "chunk_id", "pages", "cosine", "ce_score", "rerank_rank"]]

all invariants hold


,initial_rank,chunk_id,pages,cosine,ce_score,rerank_rank
0,1,C052,p.7,0.628545,8.729931,1
1,2,C185,p.19,0.543192,1.537908,3
2,3,C048,p.7,0.535401,-0.369539,8
3,4,C097,p.11,0.515184,0.432314,4
4,5,C099,p.11,0.510916,-0.058537,7
5,6,C136,p.15,0.509012,-0.042121,6
6,7,C086,p.10,0.505840,-1.813159,9
7,8,C041,p.6,0.499784,-0.038821,5
8,9,C039,p.6,0.495746,-3.317572,10
9,10,C054,p.7,0.491662,3.573735,2


## 9. Candidate-query search for failure cases

Runs all 16 queries in `experiments/candidate_queries.json`. Each has a ground-truth sentence copied from the PDF. A query is **IMPROVED** if the chunk containing that sentence is outside the baseline Top-3 but inside the re-ranked Top-3, and **REGRESSED** in the reverse case. Answers are generated for the improved and regressed queries. The table and metrics below are the raw results behind report §4.

In [18]:
# === Implementation: Ground truth and outcome labels ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

def load_candidates(path: str) -> List[Dict]:
    with open(path) as f:
        return json.load(f)


def _norm(s: str) -> str:
    # The PDF uses curly apostrophes and line-break hyphens; compare on a
    # lower-cased, apostrophe-normalised, whitespace-collapsed form.
    s = s.replace("’", "'").replace("- ", "-").lower()
    return re.sub(r"\s+", " ", s).strip()


def evidence_chunks(chunks: Sequence[Chunk], evidence: Sequence[str]) -> List[str]:
    """IDs of every chunk that contains at least one evidence sentence verbatim.

    Ground truth is defined by sentences copied from the PDF, not by chunk IDs,
    so it stays valid if the chunking configuration changes.
    """
    needles = [_norm(e) for e in evidence]
    return [c.chunk_id for c in chunks if any(n in _norm(c.text) for n in needles)]


def label_outcome(result: Dict, relevant: Sequence[str], final_k: int) -> Dict:
    """Where did the relevant evidence land in each pipeline?"""
    # relevant chunks that made it into the shared Top-10 (the reranker can only see these)
    rel = [c for c in result["candidates"] if c["chunk_id"] in set(relevant)]
    best_init = min(rel, key=lambda c: c["initial_rank"]) if rel else None
    best_rr = min(rel, key=lambda c: c["rerank_rank"]) if rel else None
    # did the LLM receive the evidence?  (Top-3 of each pipeline)
    base_hit = any(c in result["baseline_top3"] for c in relevant)
    rr_hit = any(c in result["rerank_top3"] for c in relevant)
    if not rel:
        outcome = "not in Top-10 (reranker cannot help)"
    elif rr_hit and not base_hit:
        outcome = "IMPROVED: promoted into Top-3"
    elif base_hit and not rr_hit:
        outcome = "REGRESSED: pushed out of Top-3"
    elif base_hit and rr_hit:
        outcome = "both hit"
    else:
        outcome = "both miss (in Top-10, outside both Top-3)"
    return {
        "relevant_chunks": list(relevant),
        "best_relevant_chunk": best_rr["chunk_id"] if best_rr else None,
        "initial_rank": best_rr["initial_rank"] if best_rr else None,
        "cosine": round(best_rr["cosine"], 4) if best_rr else None,
        "ce_score": round(best_rr["ce_score"], 4) if best_rr else None,
        "rerank_rank": best_rr["rerank_rank"] if best_rr else None,
        "best_initial_rank_any_relevant": best_init["initial_rank"] if best_init else None,
        "baseline_hit": base_hit,
        "rerank_hit": rr_hit,
        "outcome": outcome,
    }

In [19]:
candidates = load_candidates("experiments/candidate_queries.json")
API_BACKEND = cfg.llm_backend in {"gemini", "openai", "anthropic"}
# Free API tiers have small daily quotas: generate only for the key queries there.
GENERATE_FOR = {"Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q08", "Q16"} if API_BACKEND else None
results, rows = {}, []
for q in candidates:
    rel = evidence_chunks(chunks, q["evidence"])
    res = system.compare(q["query"], generate=GENERATE_FOR is None or q["id"] in GENERATE_FOR)
    res.update({"id": q["id"], "category": q["category"], "evidence": q["evidence"],
                "evidence_page": q["page"], "relevant_chunks": rel})
    res["label"] = label_outcome(res, rel, cfg.final_k)
    results[q["id"]] = res
    L = res["label"]
    rows.append({"id": q["id"], "category": q["category"], "query": q["query"],
                 "relevant": ",".join(rel), "page": q["page"], "chunk": L["best_relevant_chunk"],
                 "init_rank": L["initial_rank"], "rerank_rank": L["rerank_rank"],
                 "cosine": L["cosine"], "ce_score": L["ce_score"],
                 "baseline_top3": ",".join(res["baseline_top3"]), "rerank_top3": ",".join(res["rerank_top3"]),
                 "outcome": L["outcome"]})
    print(f"{q['id']} done: {L['outcome']}")

summary = pd.DataFrame(rows)
save_json(results, "results/candidate_results.json")
summary.to_csv("results/candidate_summary.csv", index=False)
summary[["id", "category", "chunk", "page", "init_rank", "rerank_rank", "cosine", "ce_score", "baseline_top3", "rerank_top3", "outcome"]]

Q01 done: IMPROVED: promoted into Top-3


Q02 done: IMPROVED: promoted into Top-3


Q03 done: IMPROVED: promoted into Top-3


Q04 done: IMPROVED: promoted into Top-3


Q05 done: IMPROVED: promoted into Top-3


...

Q13 done: both hit


Q14 done: both hit


Q15 done: both hit


Q16 done: IMPROVED: promoted into Top-3


,id,category,chunk,page,init_rank,rerank_rank,cosine,ce_score,baseline_top3,rerank_top3,outcome
0,Q01,Keyword trap / lexical distractor (+ negation),C095,11,4,1,0.5167,-0.9763,"C074,C068,C065","C095,C068,C077",IMPROVED: promoted into Top-3
1,Q02,Exceptions & corner cases (penalty clause),C014,3,6,1,0.4813,5.5618,"C032,C013,C030","C014,C030,C032",IMPROVED: promoted into Top-3
2,Q03,"Exceptions & corner cases (penalty clause, rephrased)",C014,3,6,1,0.3925,4.1107,"C032,C138,C013","C014,C032,C006",IMPROVED: promoted into Top-3
3,Q04,Exceptions & corner cases (adopted-child exception),C100,11,5,1,0.6160,7.2851,"C104,C181,C182","C100,C104,C102",IMPROVED: promoted into Top-3
4,Q05,Negation & constraints (role limit in footnote),C089,11,5,1,0.6414,7.2290,"C213,C088,C086","C089,C214,C086",IMPROVED: promoted into Top-3
5,Q06,Other document-specific (rule/exception split across chunks),C096,11,8,3,0.5909,8.3225,"C214,C097,C104","C097,C213,C096",IMPROVED: promoted into Top-3
6,Q07,Negation / constraint,C249,8,9,1,0.5870,8.2927,"C248,C056,C061","C249,C248,C056",both hit
7,Q08,Numeric threshold (control),C060,8,1,4,0.6260,2.6098,"C060,C062,C061","C062,C063,C252",REGRESSED: pushed out of Top-3
8,Q09,Role restriction,C133,15,3,2,0.5735,2.8575,"C140,C115,C133","C187,C133,C068",both hit
9,Q10,Negation / constraint (control),C254,26,2,1,0.6788,9.0178,"C253,C254,C002","C254,C253,C002",both hit


In [20]:
print(summary.outcome.value_counts().to_string())
# Aggregate retrieval quality over all queries, using the PDF-sentence labels.
# Hit@3 = relevant chunk in the Top-3 handed to the LLM; MRR@10 = 1 / rank of the first relevant chunk.
def first_rank(r, key):
    ranks = [c[key] for c in r["candidates"] if c["chunk_id"] in r["relevant_chunks"]]
    return min(ranks) if ranks else None
metrics = []
for name, key in [("baseline (bi-encoder)", "initial_rank"), ("re-ranked (Cross-Encoder)", "rerank_rank")]:
    fr = [first_rank(r, key) for r in results.values()]
    metrics.append({"pipeline": name, "Hit@1": sum(1 for x in fr if x and x <= 1), "Hit@3": sum(1 for x in fr if x and x <= 3),
                    "Hit@10": sum(1 for x in fr if x), "MRR@10": round(sum(1 / x for x in fr if x) / len(fr), 3), "queries": len(fr)})
metrics = pd.DataFrame(metrics); metrics.to_csv("results/retrieval_metrics.csv", index=False)
metrics

outcome
both hit                          8
IMPROVED: promoted into Top-3     7
REGRESSED: pushed out of Top-3    1


,pipeline,Hit@1,Hit@3,Hit@10,MRR@10,queries
0,baseline (bi-encoder),7,9,16,0.574,16
1,re-ranked (Cross-Encoder),12,15,16,0.849,16


Which query satisfies which assignment category, with the verdicts: report §4.

## 10. Failure cases

For each case: the ground-truth sentence, the full Top-10 with both rankings, and Gemini's two answers.

In [21]:
chunk_by_id = {c.chunk_id: c for c in chunks}

def show_case(qid):
    r = results[qid]; L = r["label"]
    print(f"QUERY [{qid}] ({r['category']}): {r['query']}")
    print(f"Ground-truth sentence (p.{r['evidence_page']}): \"{r['evidence'][0]}\"")
    print(f"Relevant chunk(s): {r['relevant_chunks']}  ->  best: {L['best_relevant_chunk']} "
          f"rank #{L['initial_rank']} -> #{L['rerank_rank']}  (cos={L['cosine']}, CE={L['ce_score']})")
    print(f"Baseline Top-3 : {r['baseline_top3']}")
    print(f"Reranked Top-3 : {r['rerank_top3']}\n")
    df = pd.DataFrame(r["candidates"])[["initial_rank", "chunk_id", "pages", "cosine", "ce_score", "rerank_rank"]]
    df["relevant"] = df.chunk_id.isin(r["relevant_chunks"]).map({True: "◀ relevant", False: ""})
    df["text"] = [chunk_by_id[c].text[:90] + "…" for c in df.chunk_id]
    display(df)
    for name in ("baseline", "rerank"):
        print(f"\n----- Answer {'1 (baseline)' if name == 'baseline' else '2 (reranked)'} -----")
        print(r[f"answer_{name}"])

def show_chunk(cid):
    c = chunk_by_id[cid]; print(f"{cid} ({c.pages}):\n{c.text}")

### Q16: Keyword trap — synonym (report §4.2)

In [22]:
show_case("Q16")

QUERY [Q16] (Keyword trap / synonym): Can I claim my live-in nanny who works for me as a dependent?
Ground-truth sentence (p.11): "If these people work for you, you can't claim them as dependents"
Relevant chunk(s): ['C095']  ->  best: C095 rank #4 -> #1  (cos=0.5469, CE=2.9665)
Baseline Top-3 : ['C068', 'C182', 'C088']
Reranked Top-3 : ['C095', 'C068', 'C160']



,initial_rank,chunk_id,pages,cosine,ce_score,rerank_rank,relevant,text
0,1,C068,p.8,0.583575,0.375052,2,,Your spouse is considered to live in your home even if your spouse is temporarily absent d…
1,2,C182,p.19,0.558434,-0.302626,4,,"no U.S. income, so your parent isn't a ""taxpayer."" Your children aren't your qualifying ch…"
2,3,C088,p.10,0.547007,-1.412347,5,,live with you more than half the year not a qualifying person. your relative isn't related…
3,4,C095,p.11,0.546903,2.966517,1,◀ relevant,"qualifying child, or A qualifying relative. The terms ""qualifying child"" and ""qualifying r…"
4,5,C074,p.9,0.541139,-2.663880,9,,"child and, because the child is your friend's qualifying child, your friend's child isn't …"
5,6,C160,p.17,0.529716,0.312716,3,,"The facts are the same as in Example 1, except you are only 18 years old and didn't provid…"
6,7,C104,p.12,0.525899,-1.540239,6,,a dependent unless that joint return is filed only to claim a refund of withheld income ta…
7,8,C092,p.11,0.521265,-1.584756,7,,"who lives with you and whom you can claim as a dependent. For 2023, you were entitled to f…"
8,9,C106,p.12,0.520836,-6.672370,10,,"child of more than one person, generally only one person can actually treat the child as a…"
9,10,C187,p.19,0.515938,-2.241280,8,,of your household except for any required hospital stay following birth. The test is also …



----- Answer 1 (baseline) -----
Based on the provided context, the passages do not contain information regarding whether a live-in nanny who works for you can be claimed as a dependent. Therefore, the provided context is insufficient to answer your question.

----- Answer 2 (reranked) -----
Based on the provided context, no, you cannot claim a housekeeper, maid, or servant who works for you as a dependent (p. 11).


### Q01: Keyword trap — lexical distractor (report §4.2)

In [23]:
show_case("Q01")

QUERY [Q01] (Keyword trap / lexical distractor (+ negation)): Is a housekeeper who lives in my home my dependent?
Ground-truth sentence (p.11): "If these people work for you, you can't claim them as dependents"
Relevant chunk(s): ['C095']  ->  best: C095 rank #4 -> #1  (cos=0.5167, CE=-0.9763)
Baseline Top-3 : ['C074', 'C068', 'C065']
Reranked Top-3 : ['C095', 'C068', 'C077']



,initial_rank,chunk_id,pages,cosine,ce_score,rerank_rank,relevant,text
0,1,C074,p.9,0.597620,-6.435863,4,,"child and, because the child is your friend's qualifying child, your friend's child isn't …"
1,2,C068,p.8,0.586244,-2.619330,2,,Your spouse is considered to live in your home even if your spouse is temporarily absent d…
2,3,C065,p.8,0.523435,-8.662207,6,,extensions) of the return to make the change. See Pub. 559 for more information on filing …
3,4,C095,p.11,0.516724,-0.976316,1,◀ relevant,"qualifying child, or A qualifying relative. The terms ""qualifying child"" and ""qualifying r…"
4,5,C077,p.9,0.513564,-6.076603,3,,"parent as a dependent, you can file as head of household. Temporary absences. You and your…"
5,6,C070,pp.8-9,0.500424,-10.270401,8,,spouse was a nonresident alien at any time during the year and you don't choose to treat y…
6,7,C185,p.19,0.489928,-10.403947,9,,"court of competent jurisdiction. Joint return. If you file a joint return, the person can …"
7,8,C088,p.10,0.486794,-7.717326,5,,live with you more than half the year not a qualifying person. your relative isn't related…
8,9,C075,p.9,0.486374,-9.757710,7,,entire year for your parent. If you pay more than half the cost of keeping your parent in …
9,10,C078,p.9,0.474927,-10.446819,10,,"home for this person in 2025, the person was lawfully placed with you for legal adoption b…"



----- Answer 1 (baseline) -----
Based on the provided context passages, there is no information provided about whether a housekeeper who lives in your home qualifies as a dependent. The provided context is insufficient to answer this question.

----- Answer 2 (reranked) -----
Based on the provided context, no. If housekeepers, maids, or servants work for you, you cannot claim them as dependents (p. 11).


### Q02: Exceptions & corner cases — penalty clause (report §4.3)

In [24]:
show_case("Q02")

QUERY [Q02] (Exceptions & corner cases (penalty clause)): What happens if I willfully fail to file a tax return?
Ground-truth sentence (p.3): "If you willfully fail to file a return, you may be subject to criminal prosecution"
Relevant chunk(s): ['C014']  ->  best: C014 rank #6 -> #1  (cos=0.4813, CE=5.5618)
Baseline Top-3 : ['C032', 'C013', 'C030']
Reranked Top-3 : ['C014', 'C030', 'C032']



,initial_rank,chunk_id,pages,cosine,ce_score,rerank_rank,relevant,text
0,1,C032,p.5,0.522730,-3.764716,3,,of the filing requirement in Table 1 or Table 2 that applies to you. Box 1e on Form 1099-B…
1,2,C013,pp.2-3,0.520606,-6.408201,8,,"but not losses, reported on Form 8949 or Schedule D. Gross income from a business means, f…"
2,3,C030,p.5,0.505913,-0.501732,2,,"Interest and Dividends, and its instructions. Other Situations You may have to file a tax …"
3,4,C039,p.6,0.502128,-4.247356,5,,"filed your original tax return early (for example, March 1), your return is considered fil…"
4,5,C016,p.3,0.482174,-6.236143,7,,"either case, you must add any income from investments and from incidental or outside opera…"
5,6,C014,p.3,0.481297,5.561846,1,◀ relevant,by Custodial Parent 8814 Parents' Election To Report Child's Interest and Dividends Who Mu…
6,7,C038,p.6,0.471903,-4.099555,4,,"tax year, you and your spouse must file as married individuals in both years. Annulled mar…"
7,8,C085,p.10,0.443583,-7.780656,10,,"How to file. Select this filing status by checking the ""Qualifying surviving spouse"" box o…"
8,9,C024,pp.3-4,0.439711,-5.788634,6,,"due on this income, the parent is liable for the tax. Unearned income. Unearned income inc…"
9,10,C084,p.10,0.437358,-7.354073,9,,you otherwise qualify to use that status. The year of death is the last year for which you…



----- Answer 1 (baseline) -----
Based on the provided context passages, there is no information provided about what happens if you willfully fail to file a tax return. The provided context is insufficient to answer this question.

----- Answer 2 (reranked) -----
If you willfully fail to file a tax return, you may be subject to criminal prosecution (p. 3).


### Q05: Negation & constraints — role limit in a footnote (report §4.4)

In [25]:
show_case("Q05")

QUERY [Q05] (Negation & constraints (role limit in footnote)): Is a person I can claim only under a multiple support agreement a qualifying person for head of household?
Ground-truth sentence (p.11): "If you can claim a person as a dependent only because of a multiple support agreement, that person isn't a qualifying person"
Relevant chunk(s): ['C089']  ->  best: C089 rank #5 -> #1  (cos=0.6414, CE=7.229)
Baseline Top-3 : ['C213', 'C088', 'C086']
Reranked Top-3 : ['C089', 'C214', 'C086']



,initial_rank,chunk_id,pages,cosine,ce_score,rerank_rank,relevant,text
0,1,C213,p.21,0.690668,3.812020,4,,"together provide more than half of the person's support. When this happens, you can agree …"
1,2,C088,p.10,0.667774,3.497038,5,,live with you more than half the year not a qualifying person. your relative isn't related…
2,3,C086,p.10,0.655681,3.819432,3,,for the year your spouse died. It doesn't matter whether you actually filed a joint return…
3,4,C214,p.21,0.651080,4.889964,2,,for this purpose. You can claim someone as a dependent under a multiple support agreement …
4,5,C089,pp.10-11,0.641432,7.229024,1,◀ relevant,child who is your qualifying child only because of the rules described under Children of d…
5,6,C106,p.12,0.629180,1.145254,8,,"child of more than one person, generally only one person can actually treat the child as a…"
6,7,C065,p.8,0.613595,0.738850,10,,extensions) of the return to make the change. See Pub. 559 for more information on filing …
7,8,C074,p.9,0.608810,2.617947,6,,"child and, because the child is your friend's qualifying child, your friend's child isn't …"
8,9,C069,p.8,0.593244,1.898085,7,,in Support Test for Children of Divorced or Separated Parents (or Parents Who Live Apart) …
9,10,C075,p.9,0.585027,0.850068,9,,entire year for your parent. If you pay more than half the cost of keeping your parent in …



----- Answer 1 (baseline) -----
Based on the provided context, the passages do not state whether a person claimed as a dependent only under a multiple support agreement qualifies a taxpayer for the head of household filing status. Therefore, the provided context is insufficient to answer this question. (p. 10, 21)

----- Answer 2 (reranked) -----
No. A person that you can claim as a dependent only because of a multiple support agreement is **not** a qualifying person for head of household filing status (pp. 10-11).


### Q04: Exceptions & corner cases — adopted-child exception (report §4.3)

In [26]:
show_case("Q04")

QUERY [Q04] (Exceptions & corner cases (adopted-child exception)): Does a person who is not a U.S. citizen qualify as a dependent?
Ground-truth sentence (p.11): "Exception for adopted child"
Relevant chunk(s): ['C100']  ->  best: C100 rank #5 -> #1  (cos=0.616, CE=7.2851)
Baseline Top-3 : ['C104', 'C181', 'C182']
Reranked Top-3 : ['C100', 'C104', 'C102']



,initial_rank,chunk_id,pages,cosine,ce_score,rerank_rank,relevant,text
0,1,C104,p.12,0.669401,7.141881,2,,a dependent unless that joint return is filed only to claim a refund of withheld income ta…
1,2,C181,pp.18-19,0.655593,6.130527,4,,"the qualifying child of any other taxpayer, the child is your qualifying relative as long …"
2,3,C182,p.19,0.633751,4.296385,6,,"no U.S. income, so your parent isn't a ""taxpayer."" Your children aren't your qualifying ch…"
3,4,C102,p.12,0.633517,6.587700,3,,"meet this test. You can't claim them as dependents. However, if you provided a home for a …"
4,5,C100,p.11,0.616048,7.285122,1,◀ relevant,refund of income tax withheld or estimated tax paid. The exception to the joint return tes…
5,6,C101,pp.11-12,0.615873,2.858195,7,,all year. This exception also applies if the child was lawfully placed with you for legal …
6,7,C180,p.18,0.602421,4.953856,5,,"(your friend), so you can't claim your friend's child as your qualifying relative. Also, y…"
7,8,C069,p.8,0.581177,-2.129978,10,,in Support Test for Children of Divorced or Separated Parents (or Parents Who Live Apart) …
8,9,C160,p.17,0.579362,-0.314727,8,,"The facts are the same as in Example 1, except you are only 18 years old and didn't provid…"
9,10,C068,p.8,0.569235,-1.822798,9,,Your spouse is considered to live in your home even if your spouse is temporarily absent d…



----- Answer 1 (baseline) -----
Based on the provided context, a person who is not a U.S. citizen can qualify as a dependent if they meet specific conditions:

1. **General Requirement:** A person must be a U.S. citizen, a U.S. resident alien, a U.S. national, or a resident of Canada or Mexico (p. 12, 18).
2. **Adoption Exception:** An exception exists for certain adopted children who live with you all year, even if they live in a foreign country other than Canada or Mexico (p. 18). 

Additionally, to qualify, the person must be your qualifying child or qualifying relative and meet all applicable dependency tests (such as the gross income and support tests) (p. 12, 18-19).

----- Answer 2 (reranked) -----
Based on the provided passages, a person who is not a U.S. citizen **can** qualify as a dependent if they meet specific criteria:

1. **General Requirement:** The person must be a U.S. resident alien, a U.S. national, or a resident of Canada or Mexico (p. 11, p. 12). 
2. **Exception 

### Q06: Other document-specific — rule/exception split across chunks (report §4.5)

In [27]:
show_case("Q06")

QUERY [Q06] (Other document-specific (rule/exception split across chunks)): Can a dependent claim another person as a dependent?
Ground-truth sentence (p.11): "If you can be claimed as a dependent by another taxpayer, you can't claim anyone else as a dependent"
Relevant chunk(s): ['C096']  ->  best: C096 rank #8 -> #3  (cos=0.5909, CE=8.3225)
Baseline Top-3 : ['C214', 'C097', 'C104']
Reranked Top-3 : ['C097', 'C213', 'C096']



,initial_rank,chunk_id,pages,cosine,ce_score,rerank_rank,relevant,text
0,1,C214,p.21,0.662201,7.735199,4,,for this purpose. You can claim someone as a dependent under a multiple support agreement …
1,2,C097,p.11,0.641399,8.709242,1,,can't claim any dependents on your joint return. Exception. If you can be claimed as a dep…
2,3,C104,p.12,0.636638,6.386979,6,,a dependent unless that joint return is filed only to claim a refund of withheld income ta…
3,4,C213,p.21,0.614426,8.482689,2,,"together provide more than half of the person's support. When this happens, you can agree …"
4,5,C216,pp.21-22,0.604827,5.243516,8,,"you, 24% from a relative, and 11% from a friend. Either you or the relative can claim your…"
5,6,C087,p.10,0.603653,5.916006,7,,a dependent a qualifying person. the child is married and you can't claim the child as a d…
6,7,C103,p.12,0.597409,6.597234,5,,"five tests to be the qualifying child of more than one person, there are rules you must us…"
7,8,C096,p.11,0.590904,8.322456,3,◀ relevant,"relative. For more information, see the Instructions for Form 1040. Exceptions Even if you…"
8,9,C160,p.17,0.583922,4.379674,9,,"The facts are the same as in Example 1, except you are only 18 years old and didn't provid…"
9,10,C068,p.8,0.582207,3.322763,10,,Your spouse is considered to live in your home even if your spouse is temporarily absent d…



----- Answer 1 (baseline) -----
Based on the provided context, as a general rule, if you can be claimed as a dependent by another taxpayer, you cannot claim any dependents yourself (p. 11).

**Exception:** 
You can claim someone else as a dependent if the person who can claim you (or your spouse, if filing a joint return) files a tax return **only** to claim a refund of income tax withheld or estimated tax paid (p. 11). 

To claim someone as a dependent, the person must also meet all general eligibility rules, such as being a qualifying child or qualifying relative, meeting the joint return rules, and meeting citizenship/residency requirements (p. 12).

----- Answer 2 (reranked) -----
Based on the provided context, generally **no**, you cannot claim anyone else as a dependent if you (or your spouse, if filing a joint return) can be claimed as a dependent by another taxpayer, even if you have a qualifying child or qualifying relative (p. 11).

**Exception:** 
You can claim someone else

### Regression (Q08) and controls (report §4.6)

In [28]:
for qid in ["Q03", "Q08", "Q07", "Q12"]:
    r = results[qid]
    print(f"[{qid}] {r['query']}\n  outcome: {r['label']['outcome']} | rank {r['label']['initial_rank']} -> {r['label']['rerank_rank']}")
    print(f"  baseline {r['baseline_top3']}  reranked {r['rerank_top3']}")
    if "answer_baseline" in r:
        print("  A1:", r["answer_baseline"][:500].replace("\n", " "))
        print("  A2:", r["answer_rerank"][:500].replace("\n", " "), "\n")
    else:
        print("  (retrieval only in this run)\n")

[Q03] What is the penalty for failing to file a required return?
  outcome: IMPROVED: promoted into Top-3 | rank 6 -> 1
  baseline ['C032', 'C138', 'C013']  reranked ['C014', 'C032', 'C006']
  A1: Based on the provided context passages, there is no information provided about the penalty for failing to file a required return. The provided context is insufficient to answer this question.
  A2: Based on the provided context, if you are required to file a return but fail to do so, you may have to pay a penalty, and if you willfully fail to file a return, you may be subject to criminal prosecution (p. 3). 

[Q08] What is the capital loss deduction limit for married filing separately?
  outcome: REGRESSED: pushed out of Top-3 | rank 1 -> 4
  baseline ['C060', 'C062', 'C061']  reranked ['C062', 'C063', 'C252']
  A1: Based on the provided context, the capital loss deduction limit for married individuals filing separately is $1,500 (instead of $3,000 on a joint return) (p. 8).
  A2: Based on th

## 11. Latency profiling

`time.perf_counter()`, models pre-loaded, 3 warm-up calls, then 20 runs × 16 queries per stage. Generation times are those of the original API calls (network included, rate-limit waits excluded). Method and interpretation: report §3 and §5.

In [29]:
# === Implementation: Latency ===
# (Identical to the matching section of src/; tests/test_pipeline.py checks this code
#  and tests/test_notebook_sync.py checks the notebook has not drifted from it.)

def _stats(xs: Sequence[float]) -> Dict:
    """Summary statistics of a list of durations (seconds in, milliseconds out)."""
    ms = [x * 1000 for x in xs]
    return {"n": len(ms), "mean_ms": statistics.mean(ms), "median_ms": statistics.median(ms),
            "stdev_ms": statistics.stdev(ms) if len(ms) > 1 else 0.0,
            "min_ms": min(ms), "max_ms": max(ms)}


def profile_retrieval(system: RAGSystem, queries: Sequence[str], runs: int = 20, warmup: int = 3) -> Dict:
    """Time Stage 1 (query embedding + FAISS search) and Stage 2 (Cross-Encoder
    over the same 10 candidates) separately.  Models are already loaded, so
    load time is excluded; warm-up calls are discarded."""
    # warm-up: first calls pay one-off costs (memory allocation, thread start-up)
    for q in queries[:1] * warmup:
        c, _ = system.retrieve(q)
        system.rerank(q, c)
    # measured runs: both stages see exactly the same query and candidate list
    ret, rr, sel = [], [], []
    for _ in range(runs):
        for q in queries:
            c, t_ret = system.retrieve(q)
            t0 = time.perf_counter()
            select_top(c, system.cfg.final_k)
            sel.append(time.perf_counter() - t0)
            _, t_rr = system.rerank(q, c)
            ret.append(t_ret)
            rr.append(t_rr)
    return {"retrieval_top10": _stats(ret), "baseline_select_top3": _stats(sel),
            "crossencoder_rerank_10": _stats(rr),
            "retrieval_plus_rerank": _stats([a + b for a, b in zip(ret, rr)])}


def profile_generation(system: RAGSystem, queries: Sequence[str], runs: int = 2, warmup: int = 1) -> Dict:
    """End-to-end timings for both pipelines (retrieval -> [rerank] -> LLM)."""
    q0 = queries[0]
    for _ in range(warmup):
        c, _ = system.retrieve(q0)
        system.llm.generate(build_prompt(q0, select_top(c, system.cfg.final_k)))
    rows = []
    for _ in range(runs):
        for q in queries:
            t0 = time.perf_counter()
            c, t_ret = system.retrieve(q)
            base_ctx = select_top(c, system.cfg.final_k)
            a1 = system.llm.generate(build_prompt(q, base_ctx))
            t_gen1 = system.llm.last_call_s
            # rate-limit pacing / retries (API backends) are excluded from end-to-end time
            t_e2e_base = time.perf_counter() - t0 - system.llm.last_wait_s

            t0 = time.perf_counter()
            c, t_ret2 = system.retrieve(q)
            rr, t_rr = system.rerank(q, c)
            rr_ctx = select_top(rr, system.cfg.final_k)
            a2 = system.llm.generate(build_prompt(q, rr_ctx))
            t_gen2 = system.llm.last_call_s
            t_e2e_rr = time.perf_counter() - t0 - system.llm.last_wait_s
            rows.append({"query": q, "retrieval_s": t_ret, "generation_baseline_s": t_gen1,
                         "e2e_baseline_s": t_e2e_base, "retrieval_s_2": t_ret2, "rerank_s": t_rr,
                         "generation_rerank_s": t_gen2, "e2e_rerank_s": t_e2e_rr,
                         "answer_len_baseline": len(a1), "answer_len_rerank": len(a2)})
    summary = {k: _stats([r[k] for r in rows]) for k in
               ("generation_baseline_s", "generation_rerank_s", "e2e_baseline_s", "e2e_rerank_s")}
    return {"rows": rows, "summary": summary}

In [30]:
lat = profile_retrieval(system, [q["query"] for q in candidates], runs=20, warmup=3)
lat_df = pd.DataFrame(lat).T[["n", "mean_ms", "median_ms", "stdev_ms", "min_ms", "max_ms"]].round(2)
lat_df

,n,mean_ms,median_ms,stdev_ms,min_ms,max_ms
retrieval_top10,320.0,10.21,8.89,2.30,7.77,17.28
baseline_select_top3,320.0,0.00,0.00,0.00,0.00,0.01
crossencoder_rerank_10,320.0,267.05,260.16,25.41,231.78,331.15
retrieval_plus_rerank,320.0,277.26,270.53,25.82,240.04,344.04


In [31]:
if API_BACKEND:
    gq = [results[q] for q in sorted(GENERATE_FOR)]
    rows_ = [{"query": r["query"], "generation_baseline_s": r["timing_s"]["generation_baseline"],
              "generation_rerank_s": r["timing_s"]["generation_rerank"],
              "e2e_baseline_s": r["timing_s"]["retrieval"] + r["timing_s"]["generation_baseline"],
              "e2e_rerank_s": r["timing_s"]["retrieval"] + r["timing_s"]["rerank"] + r["timing_s"]["generation_rerank"]} for r in gq]
    gen = {"rows": rows_, "summary": {k: _stats([x[k] for x in rows_]) for k in
           ("generation_baseline_s", "generation_rerank_s", "e2e_baseline_s", "e2e_rerank_s")}}
else:
    gen = profile_generation(system, [results[q]["query"] for q in ["Q01", "Q02", "Q05"]], runs=2, warmup=1)
gen_df = pd.DataFrame(gen["summary"]).T[["n", "mean_ms", "median_ms", "stdev_ms", "min_ms", "max_ms"]]
gen_df = (gen_df[["mean_ms", "median_ms", "stdev_ms", "min_ms", "max_ms"]] / 1000).round(2).rename(columns=lambda c: c.replace("_ms", "_s"))
gen_df.insert(0, "n", pd.DataFrame(gen["summary"]).T["n"].astype(int))
save_json({"env": env_info, "retrieval_ms": lat, "generation": gen}, "results/latency.json")
gen_df

,n,mean_s,median_s,stdev_s,min_s,max_s
generation_baseline_s,8,3.22,3.20,1.29,1.73,5.99
generation_rerank_s,8,6.52,2.43,12.06,1.50,36.34
e2e_baseline_s,8,3.23,3.21,1.29,1.74,6.00
e2e_rerank_s,8,6.79,2.74,12.07,1.77,36.65


In [32]:
r_ms, rr_ms = lat["retrieval_top10"]["median_ms"], lat["crossencoder_rerank_10"]["median_ms"]
table = pd.DataFrame([
    {"stage": "Stage 1: embed query + FAISS Top-10", "baseline": r_ms, "re-ranking": r_ms},
    {"stage": "Stage 2: Cross-Encoder on 10 pairs", "baseline": 0.0, "re-ranking": rr_ms},
    {"stage": "Retrieval total (ms, median)", "baseline": r_ms, "re-ranking": r_ms + rr_ms},
    {"stage": "LLM generation (s, median)", "baseline": gen["summary"]["generation_baseline_s"]["median_ms"] / 1000,
     "re-ranking": gen["summary"]["generation_rerank_s"]["median_ms"] / 1000},
    {"stage": "End-to-end (s, median)", "baseline": gen["summary"]["e2e_baseline_s"]["median_ms"] / 1000,
     "re-ranking": gen["summary"]["e2e_rerank_s"]["median_ms"] / 1000},
]).round(2)
print(f"Re-ranking adds {rr_ms:.1f} ms to retrieval: {(r_ms + rr_ms) / r_ms:.1f}x the baseline retrieval time, "
      f"i.e. {100 * rr_ms / 1000 / (gen['summary']['e2e_rerank_s']['median_ms'] / 1000):.2f}% of the re-ranking pipeline's end-to-end time with {cfg.llm_model}.")
table.to_csv("results/latency_table.csv", index=False)
table

Re-ranking adds 260.2 ms to retrieval: 30.3x the baseline retrieval time, i.e. 9.50% of the re-ranking pipeline's end-to-end time with gemini-3.5-flash.


,stage,baseline,re-ranking
0,Stage 1: embed query + FAISS Top-10,8.89,8.89
1,Stage 2: Cross-Encoder on 10 pairs,0.00,260.16
2,"Retrieval total (ms, median)",8.89,269.05
3,"LLM generation (s, median)",3.20,2.43
4,"End-to-end (s, median)",3.21,2.74


## 12. Scaling question — why not Cross-Encode the whole corpus?

Measures Cross-Encoding **all** chunks for one query and extrapolates to larger corpora. Discussion: report §6.

In [33]:
q = results["Q01"]["query"]
all_texts = [c.text for c in chunks]
reranker.score(q, all_texts[:16])                     # warm-up
full = []
for _ in range(3):
    t0 = time.perf_counter(); s_all = reranker.score(q, all_texts); full.append(time.perf_counter() - t0)
full_ms = float(np.median(full) * 1000)
per_pair = full_ms / len(chunks)
best_full = chunks[int(np.argmax(s_all))].chunk_id
scal = {"corpus_chunks": len(chunks), "full_crossencode_ms_median": full_ms, "per_pair_ms": per_pair,
        "top10_rerank_ms_median": rr_ms, "bi_encoder_search_ms_median": r_ms,
        "best_chunk_full_corpus": best_full, "best_chunk_two_stage": results["Q01"]["rerank_top3"][0]}
for n in [1e4, 1e6, 1e8]:
    scal[f"est_full_crossencode_{int(n):,}_chunks_s"] = per_pair * n / 1000
save_json(scal, "results/scaling.json")
pd.Series(scal)

corpus_chunks                                           292
full_crossencode_ms_median                      9005.405306
per_pair_ms                                       30.840429
top10_rerank_ms_median                           260.158509
bi_encoder_search_ms_median                        8.894106
best_chunk_full_corpus                                 C095
best_chunk_two_stage                                   C095
est_full_crossencode_10,000_chunks_s             308.404291
est_full_crossencode_1,000,000_chunks_s         30840.42913
est_full_crossencode_100,000,000_chunks_s    3084042.913013
dtype: object

## 13. Output files

Everything quoted in the report is saved in `results/`.

In [34]:
# Final cross-check: everything quoted in the report is loaded from these files.
print(sorted(os.listdir("results")))

['candidate_results.json', 'candidate_summary.csv', 'chunks.json', 'latency.json', 'latency_table.csv', 'llm_cache.json', 'qwen2.5-1.5b-local', 'retrieval_metrics.csv', 'scaling.json']
